In [1]:
!pip install transformer_lens
!pip install sae_lens
!pip install transformers
!pip install torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.9 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=185c4cf38e3e8166227e0a957c42c8ff28d6ee7d2c781171bb59f03840028c42
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.4/298.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━

In [2]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer


In [3]:
prompt = "The capital of France is"

print(model.generate(prompt))

  0%|          | 0/10 [00:00<?, ?it/s]

The capital of France is also seeing rising violence and Islamist anniversary attacks like the


In [5]:
prompt = "The Eiffel Tower is located in the city of"
logits = model(prompt)
predicted_token = model.to_string(logits[0, -1].argmax())
print(predicted_token)

 London


In [6]:
from sae_lens import SAE

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gpt2-small-res-jb",   # a community-trained SAE set for GPT-2-small residual stream
    sae_id="blocks.8.hook_resid_pre",  # residual stream at layer 8
    device="cuda"
)

cfg.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

blocks.8.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.8.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/sae_lens/saes/sae.py:253: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(
/tmp/ipykernel_1728/4267268797.py:3: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [7]:
_, cache = model.run_with_cache(prompt)
activation = cache["blocks.8.hook_resid_pre"]  # shape: [batch, seq_len, d_model]

features = sae.encode(activation)   # shape: [batch, seq_len, n_features] — mostly zeros
last_token_features = features[0, -1, :]

top_features = last_token_features.topk(5)
print(top_features)

torch.return_types.topk(
values=tensor([35.0176, 31.4880,  9.5883,  5.6612,  4.0742], device='cuda:0',
       grad_fn=<TopkBackward0>),
indices=tensor([ 5856, 11149,  2194, 21062, 22852], device='cuda:0'))


In [20]:
target_feature = 11149   # whichever feature index you decide is most relevant after checking Neuronpedia

def ablate_hook(activation, hook):
    feature_acts = sae.encode(activation)               # decode into sparse feature space
    contribution = feature_acts[..., target_feature].unsqueeze(-1) * sae.W_dec[target_feature]
    return activation - contribution                     # subtract only that feature's direction

logits_ablated = model.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.8.hook_resid_pre", ablate_hook)]
)
predicted_token_ablated = model.to_string(logits_ablated[0, -1].argmax())
print("Before:", predicted_token)          # "London" from your earlier cell
print("After ablation:", predicted_token_ablated)

Before:  London
After ablation:  Paris


In [10]:
import torch.nn.functional as F

def show_top_tokens(logits, k=5):
    probs = F.softmax(logits[0, -1], dim=-1)
    top = probs.topk(k)
    for val, idx in zip(top.values, top.indices):
        print(f"{model.to_string(idx):>15}: {val.item():.4f}")

print("BEFORE:")
show_top_tokens(logits)
print("\nAFTER ablation (feature 11149):")
show_top_tokens(logits_ablated)

BEFORE:
         London: 0.0691
          Paris: 0.0688
      Amsterdam: 0.0403
         Berlin: 0.0323
            New: 0.0279

AFTER ablation (feature 11149):
          Paris: 0.0334
      Amsterdam: 0.0268
              E: 0.0251
         Berlin: 0.0220
              L: 0.0186


In [17]:
prompt2 = "The Colosseum is located in the city of"
logits2 = model(prompt2)
print("Before:", model.to_string(logits2[0, -1].argmax()))

_, cache2 = model.run_with_cache(prompt2)
activation2 = cache2["blocks.8.hook_resid_pre"]
features2 = sae.encode(activation2)
print("Feature 11149 activation on this prompt:", features2[0, -1, 11149].item())

logits2_ablated = model.run_with_hooks(
    prompt2,
    fwd_hooks=[("blocks.8.hook_resid_pre", ablate_hook)]
)
print("After ablation:", model.to_string(logits2_ablated[0, -1].argmax()))

Before:  Rome
Feature 11149 activation on this prompt: 29.280553817749023
After ablation:  C


In [15]:
print("BEFORE (Colosseum):")
show_top_tokens(logits2)
print("\nAFTER ablation:")
show_top_tokens(logits2_ablated)

BEFORE (Colosseum):
           Rome: 0.0193
              P: 0.0183
              L: 0.0166
              T: 0.0166
            Col: 0.0148

AFTER ablation:
              C: 0.0196
              T: 0.0183
              P: 0.0169
              S: 0.0158
              L: 0.0146


In [22]:
def partial_ablate_hook(activation, hook):
    feature_acts = sae.encode(activation)
    contribution = feature_acts[..., target_feature].unsqueeze(-1) * sae.W_dec[target_feature]
    return activation - 0.3 * contribution   # only 30% suppression

In [23]:
logits_partially_ablated = model.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.8.hook_resid_pre", partial_ablate_hook)]
)
predicted_token_partially_ablated = model.to_string(logits_partially_ablated[0, -1].argmax())

print("Original prediction:", predicted_token)
print("Partially ablated prediction (30% suppression):", predicted_token_partially_ablated)

Original prediction:  London
Partially ablated prediction (30% suppression):  Paris


In [25]:
print("BEFORE partial ablation:")
show_top_tokens(logits)
print("\nAFTER partial ablation (30% suppression of feature 11149):")
show_top_tokens(logits_partially_ablated)

BEFORE partial ablation:
         London: 0.0691
          Paris: 0.0688
      Amsterdam: 0.0403
         Berlin: 0.0323
            New: 0.0279

AFTER partial ablation (30% suppression of feature 11149):
          Paris: 0.0610
         London: 0.0481
      Amsterdam: 0.0385
         Berlin: 0.0298
            New: 0.0191


In [26]:
logits2_partial = model.run_with_hooks(
    prompt2,
    fwd_hooks=[("blocks.8.hook_resid_pre", partial_ablate_hook)]
)
print("BEFORE (Colosseum):")
show_top_tokens(logits2)
print("\nAFTER 30% partial ablation:")
show_top_tokens(logits2_partial)

BEFORE (Colosseum):
           Rome: 0.0193
              P: 0.0183
              L: 0.0166
              T: 0.0166
            Col: 0.0148

AFTER 30% partial ablation:
              P: 0.0184
              T: 0.0174
              L: 0.0163
              C: 0.0157
           Rome: 0.0147


In [27]:
def gentle_ablate_hook(activation, hook):
    feature_acts = sae.encode(activation)
    contribution = feature_acts[..., target_feature].unsqueeze(-1) * sae.W_dec[target_feature]
    return activation - 0.1 * contribution

logits2_gentle = model.run_with_hooks(
    prompt2,
    fwd_hooks=[("blocks.8.hook_resid_pre", gentle_ablate_hook)]
)
print("BEFORE:")
show_top_tokens(logits2)
print("\nAFTER 10% suppression:")
show_top_tokens(logits2_gentle)

BEFORE:
           Rome: 0.0193
              P: 0.0183
              L: 0.0166
              T: 0.0166
            Col: 0.0148

AFTER 10% suppression:
              P: 0.0184
           Rome: 0.0177
              T: 0.0169
              L: 0.0166
            Col: 0.0146


In [29]:
import torch
import torch.nn.functional as F
import pandas as pd

# ---- Config ----
FEATURE_ID = 11149
HOOK_NAME = "blocks.8.hook_resid_post"  # adjust if your SAE release uses a different point
ABLATION_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]  # 0 = no change, 1 = full ablation
TOP_K = 10

test_cases = [
    {"prompt": "The Eiffel Tower is in the city of", "correct": " Paris"},
    {"prompt": "The Colosseum is in the city of",     "correct": " Rome"},
    {"prompt": "Big Ben is in the city of",            "correct": " London"},
    {"prompt": "The Sagrada Familia is in the city of","correct": " Barcelona"},
    {"prompt": "The Statue of Liberty is in the city of", "correct": " New York"},
]

# ---- Core hook factory ----
def make_ablation_hook(feature_id, sae, strength):
    """
    strength=0.0 -> no change, 1.0 -> full ablation of the feature's contribution.
    Encodes activation into SAE feature space, scales down the target feature,
    then decodes back and patches the residual stream.
    """
    def hook_fn(resid, hook):
        # resid: [batch, seq, d_model]
        feature_acts = sae.encode(resid)                # [batch, seq, d_sae]
        original_feature_val = feature_acts[..., feature_id].clone()

        # Scale down just the target feature
        feature_acts[..., feature_id] = original_feature_val * (1.0 - strength)

        # Decode back to residual stream space
        reconstructed = sae.decode(feature_acts)

        # SAE reconstructions aren't perfect; patch only the delta caused by
        # our intervention rather than replacing resid wholesale.
        baseline_reconstructed = sae.decode(sae.encode(resid))
        delta = reconstructed - baseline_reconstructed
        return resid + delta

    return hook_fn


def get_distribution(prompt, strength=None, feature_id=None, sae=None):
    tokens = model.to_tokens(prompt)
    if strength is not None and strength > 0.0:
        hook_fn = make_ablation_hook(feature_id, sae, strength)
        logits = model.run_with_hooks(tokens, fwd_hooks=[(HOOK_NAME, hook_fn)])
    else:
        logits = model(tokens)
    last_logits = logits[0, -1, :]
    probs = F.softmax(last_logits, dim=-1)
    return probs


def kl_divergence(p, q, eps=1e-10):
    p = p + eps
    q = q + eps
    return torch.sum(p * torch.log(p / q)).item()


def l1_distance(p, q):
    return torch.sum(torch.abs(p - q)).item()


# ---- Main sweep ----
results = []

for case in test_cases:
    prompt = case["prompt"]
    correct_str = case["correct"]

    # Fix: Handle multi-token correct strings by taking the last token
    correct_token_ids = model.to_tokens(correct_str, prepend_bos=False).squeeze()
    if correct_token_ids.numel() > 1:
        correct_token_id = correct_token_ids[-1].item()
    else:
        correct_token_id = correct_token_ids.item()

    # Clean baseline distribution (no ablation)
    clean_probs = get_distribution(prompt)
    clean_top_id = torch.argmax(clean_probs).item()
    clean_top_str = model.to_string([clean_top_id])

    for strength in ABLATION_LEVELS:
        probs = get_distribution(prompt, strength=strength, feature_id=FEATURE_ID, sae=sae)

        top_probs, top_ids = torch.topk(probs, TOP_K)
        top_tokens = [model.to_string([tid]) for tid in top_ids]

        correct_prob = probs[correct_token_id].item()
        correct_rank = (probs > probs[correct_token_id]).sum().item()  # 0 = top rank

        results.append({
            "prompt": prompt,
            "correct_answer": correct_str,
            "ablation_strength": strength,
            "top1_token": top_tokens[0],
            "top1_prob": top_probs[0].item(),
            "correct_token_prob": correct_prob,
            "correct_token_rank": correct_rank,
            "kl_from_clean": kl_divergence(clean_probs, probs) if strength > 0 else 0.0,
            "l1_from_clean": l1_distance(clean_probs, probs) if strength > 0 else 0.0,
            "top5_tokens": top_tokens[:5],
            "top5_probs": [round(p.item(), 4) for p in top_probs[:5]],
        })

df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
print(df[["prompt", "ablation_strength", "top1_token", "correct_token_prob",
          "correct_token_rank", "kl_from_clean"]].to_string(index=False))

df.to_csv("ablation_sweep_results.csv", index=False)


                                 prompt  ablation_strength top1_token  correct_token_prob  correct_token_rank  kl_from_clean
     The Eiffel Tower is in the city of                0.0     London            0.068731                   1       0.000000
     The Eiffel Tower is in the city of                0.1     London            0.068101                   1       0.000923
     The Eiffel Tower is in the city of                0.2     London            0.067279                   1       0.003761
     The Eiffel Tower is in the city of                0.3      Paris            0.066259                   0       0.008621
     The Eiffel Tower is in the city of                0.5      Paris            0.063612                   0       0.024885
     The Eiffel Tower is in the city of                0.7      Paris            0.060159                   0       0.050791
     The Eiffel Tower is in the city of                1.0      Paris            0.053602                   0       0.110563
